In [1]:
# ============================================================
# PART 0 — Setup (Kaggle)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # before `import torch`

import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn

sns.set_style("whitegrid")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

!pip install pennylane pennylane-lightning --upgrade -q

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}  device: {DEVICE}")

# --- EDIT THESE PATHS FOR YOUR KAGGLE SESSION ---------------------
DATASET = "UNSW-NB15"
SCRIPTS_PARENT_DIR = "/kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts"         
DATA_DIR           = f"/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/FROZEN/{DATASET}"
CHECKPOINT_PATH    = "/kaggle/input/models/lawunnannda/qsentinel-models/pytorch/default/7/final-unsw-nb15-maqt-train-checkpoint.pt"
HANDOFF_PATH = Path("/kaggle/input/datasets/lawunnannda/quantum-sentinel-iot-v1-0/team-artifacts/teamC_week3_FROZEN_handoff.json")
# --------------------------------------------------------------------------


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 9.1 MB/s eta 0:00:00
CUDA available: False  device: cpu


In [2]:
if SCRIPTS_PARENT_DIR not in sys.path:
    sys.path.append(SCRIPTS_PARENT_DIR)
import scripts
print("scripts loaded from:", scripts.__file__)

from scripts.constants import (
    DEFAULT_ALPHA, DEFAULT_CF, DEFAULT_NOISE_RATE, DEFAULT_REUPLOAD, ZERO_DAY,
    DEFAULT_BETA, PROP1_RESIDUAL_TOL, INPUT_DIM_D, DEFAULT_LOWER_PERCENTILE,
    DEFAULT_UPPER_PERCENTILE,
)
from scripts.data import (
    load_split, capped_sample, greedy_dpp_sample, to_angles,
    class_balance_table, plot_class_balance_bars, plot_class_balance_pie,
)
from scripts.circuit import create_quantum_device, build_forward_circuit
from scripts.prototypes import PrototypeBank
from scripts.quantum_metrics import fidelity, fidelity_pairwise, trace_distance, stack_prototypes
from scripts.inference import qsnet_infer_batch
from scripts.conformal import (
    min_calibration_size, class_conditional_calibrate, per_class_empirical_far,
    threshold_from_scores,
)
from scripts.utils import to_np_batch_x, to_np_y, to_torch_batch_x, expectations_to_tensor
from scripts.logging import to_jsonable, append_jsonl
from scripts.theory import (
    verify_fidelity_convention, check_proposition1_real_data, assert_proposition1,
    analytic_lipschitz_bound, check_lipschitz_tightness,
    linf_to_l2_budget, l2_to_linf_budget, is_eps_safe_to_plot, clopper_pearson_ci,
    worst_case_f_in, quantile_f_out, proposition2_epsilon_star, proposition2_epsilon_beta,
    robust_f_in, robust_f_out, proposition2_epsilon_robust,
    fgsm_gradient_sign, apply_fgsm_perturbation, two_sample_discriminability_auroc,
)
from scripts.cache import (
    ForwardCache, cached_f_max, cached_nonconformity_scores, cached_calibrate_threshold,
    cached_conformal_alpha_sweep, cached_predict_labels, cached_qsnet_infer,
    cached_lipschitz_percentile, cached_qsnet_infer_per_class,
)
from scripts.hilbert import (
    hilbert_geometry_diagnostics, print_h1_report, fidelity_to_prototypes_matrix, pca_2d,
)
from scripts.memory import (
    is_oom_error, is_fatal_cuda_error, safe_empty_cache, run_batched_safely, gpu_memory_snapshot,
)
from scripts.attacks import fgsm_attack, pgd_attack, eval_attacked, robustness_ablation
from scripts.train import train_plain_vqc  # only used in the optional Day-26 ablation block

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# --- Kaggle output layout ---------------------------------------------------
OUT_ROOT   = Path("/kaggle/working/quantum-sentinel")
CACHE_DIR  = OUT_ROOT / "caches"
CKPT_DIR   = OUT_ROOT / "checkpoints"
LOG_DIR    = OUT_ROOT / "logs"
FIG_DIR    = OUT_ROOT / "figures"
TABLE_DIR  = OUT_ROOT / "tables"
for d in (CACHE_DIR, CKPT_DIR, LOG_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

def savefig(name):
    p = FIG_DIR / f"{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"  saved figure -> {p}")
    plt.close()

def savejson(obj, name, subdir=LOG_DIR):
    p = Path(subdir) / f"{name}.json"
    with open(p, "w") as f:
        json.dump(to_jsonable(obj), f, indent=2)
    print(f"  saved json -> {p}")
    return p

def savecsv(df, name, subdir=TABLE_DIR):
    p = Path(subdir) / f"{name}.csv"
    df.to_csv(p, index=False)
    print(f"  saved table -> {p}")
    return p

check = verify_fidelity_convention(dim=4, n_trials=30, seed=SEED)
print("Fidelity-convention self-check PASSED (synthetic unit test only):", check)

scripts loaded from: /kaggle/input/datasets/lawunnannda/quantum-sentinel-scripts/scripts/__init__.py
Fidelity-convention self-check PASSED (synthetic unit test only): {'max_pairwise_disagreement': 1.5543122344752192e-15, 'max_fvg_violation': 0.0}


In [3]:
# ============================================================
# PART 1 — Load checkpoint (theta_star, head, prototypes, config)
# ============================================================
print("="*60); print("LOADING CHECKPOINT"); print("="*60)
assert Path(CHECKPOINT_PATH).exists(), f"Not found: {CHECKPOINT_PATH}"
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)

theta_star      = ckpt["theta"]
head_state_dict = ckpt["head_state_dict"]
prototypes_raw  = ckpt["prototypes"]
class_names     = ckpt["class_names"]
num_classes     = ckpt["num_classes"]
num_qubits      = ckpt["num_qubits"]
num_layers      = ckpt["num_layers"]
noise_rate      = ckpt.get("noise_rate", DEFAULT_NOISE_RATE)
reupload        = ckpt.get("reupload", DEFAULT_REUPLOAD)
feature_cols    = ckpt["feature_cols"]
use_pca         = ckpt.get("use_pca", True)
target_col      = "label_multiclass"

scaler   = ckpt["scaler"]
pca      = ckpt.get("pca", None)
angle_x_min = np.asarray(ckpt["angle_x_min"])
angle_x_max = np.asarray(ckpt["angle_x_max"])
angle_max   = ckpt.get("angle_max", float(np.pi))

SUBSET        = ckpt.get("subset", True)
PER_CLASS_CAP = ckpt.get("per_class_cap", 7000)
SAMPLER       = ckpt.get("sampler", "greedy_dpp")

prototypes = {int(k): (v if torch.is_tensor(v) else torch.tensor(v)).to(DEVICE)
              for k, v in prototypes_raw.items()}

print(f" classes    : {class_names}")
print(f" qubits     : {num_qubits}  layers: {num_layers}  reupload: {reupload}  noise: {noise_rate}")
print(f" theta      : {tuple(theta_star.shape)}")
print(f" prototypes : {len(prototypes)} classes")

classifier_head = nn.Linear(num_qubits, num_classes).to(DEVICE)
classifier_head.load_state_dict(head_state_dict)
classifier_head.eval()
print(f" head       : Linear({num_qubits} -> {num_classes}) loaded OK")

LOADING CHECKPOINT
 classes    : ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance']
 qubits     : 4  layers: 2  reupload: True  noise: 0.01
 theta      : (2, 4, 3)
 prototypes : 8 classes
 head       : Linear(4 -> 8) loaded OK


In [4]:
# ============================================================
# Verify Team C's FROZEN features
# ============================================================
print("="*60); print("VERIFYING FROZEN FEATURES"); print("="*60)

handoff = json.loads(HANDOFF_PATH.read_text())
teamc_cols = list(handoff["frozen_subsets"][DATASET]["features"])
ckpt_cols = list(ckpt["feature_cols"])
print(f"Team C selector: {handoff['frozen_subsets'][DATASET]['selector']}\n")
print(f"Team C k={len(teamc_cols)}: {teamc_cols}\n")
print(f"ckpt   k={len(ckpt_cols)}: {ckpt_cols}\n")

same_set = set(teamc_cols) == set(ckpt_cols)
same_order = teamc_cols == ckpt_cols
print(f"same features (set):   {same_set}")
print(f"same order (list):     {same_order}")
print()
if not same_set:
    only_teamc = sorted(set(teamc_cols) - set(ckpt_cols))
    only_ckpt = sorted(set(ckpt_cols) - set(teamc_cols))
    print(f"  only in Team C: {only_teamc}")
    print(f"  only in ckpt:   {only_ckpt}")
    raise ValueError("checkpoint feature_cols do not match Team C FROZEN subset")
if not same_order:
    print("warning: same columns, different order — encoding may still break if order mattered at train time")

VERIFYING FROZEN FEATURES
Team C selector: RF

Team C k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'src_bytes', 'dst_bytes', 'rate', 'rate_src', 'rate_dst', 'pkt_size_mean', 'iat']

ckpt   k=10: ['duration', 'n_pkts_total', 'n_bytes_total', 'src_bytes', 'dst_bytes', 'rate', 'rate_src', 'rate_dst', 'pkt_size_mean', 'iat']

same features (set):   True
same order (list):     True



In [5]:
# ============================================================
# PART 2 — Load raw splits, apply the SAME subsampling/encoding as training
# ============================================================
print("="*60); print("LOADING RAW DATA SPLITS"); print("="*60)

X_train_full, y_train_full, _ = load_split(DATA_DIR, "train", target_col, csv=True, selected_cols=feature_cols)
X_test,       y_test,       _ = load_split(DATA_DIR, "test", target_col, csv=True, selected_cols=feature_cols)
X_cal,        y_cal,        _ = load_split(DATA_DIR, "calibration", target_col, csv=True, selected_cols=feature_cols)
X_zeroday,    y_zeroday,    _ = load_split(DATA_DIR, "zeroday", target_col, csv=True, selected_cols=feature_cols)

print(f" train(full): {X_train_full.shape}  test: {X_test.shape}  cal: {X_cal.shape}  zeroday: {X_zeroday.shape}")

if SUBSET and SAMPLER == "greedy_dpp":
    X_train, y_train = greedy_dpp_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
elif SUBSET:
    X_train, y_train = capped_sample(X_train_full, y_train_full, per_class_cap=PER_CLASS_CAP, seed=SEED)
else:
    X_train, y_train = X_train_full, y_train_full
print(f" train(subset): {X_train.shape} via {SAMPLER}")

def encode(X):
    return to_angles(X, scaler, angle_x_min, angle_x_max,
                      pca=pca if use_pca else None, angle_max=angle_max)

A_train, A_test, A_cal, A_zeroday = encode(X_train), encode(X_test), encode(X_cal), encode(X_zeroday)
print(f" A_train {A_train.shape}  A_test {A_test.shape}  A_cal {A_cal.shape}  A_zeroday {A_zeroday.shape}")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
plot_class_balance_bars(y_train, class_names, title="Train (subset) class balance", ax=ax[0])
plot_class_balance_pie(y_train, class_names, title="Train (subset) class balance", ax=ax[1])
plt.tight_layout(); savefig("part2_class_balance")

LOADING RAW DATA SPLITS
 train(full): (81215, 10)  test: (10086, 10)  cal: (10112, 10)  zeroday: (1216, 10)
 train(subset): (81215, 10) via None
 A_train (81215, 4)  A_test (10086, 4)  A_cal (10112, 4)  A_zeroday (1216, 4)
  saved figure -> /kaggle/working/quantum-sentinel/figures/part2_class_balance.png


In [6]:
# ============================================================
# PART 3 — Quantum device + ForwardCache (ONE forward pass per split, ever)
# ============================================================
print("="*60); print("QUANTUM CIRCUIT"); print("="*60)

dev = create_quantum_device(num_qubits)                 # default.mixed
forward_circuit = build_forward_circuit(dev, num_qubits, num_layers,
                                         noise_rate=noise_rate, reupload=reupload)
theta_device = theta_star.to(DEVICE)
print(f" device={DEVICE}  wires={num_qubits}  theta on {theta_device.device}")

CACHE_LATEST = CACHE_DIR / "forward_cache_latest.pt"
cache = ForwardCache(store_device="cpu")

splits = [("train", A_train), ("test", A_test), ("cal", A_cal), ("zeroday", A_zeroday)]

if CACHE_LATEST.exists():
    print(f"RESUME: loading forward cache from {CACHE_LATEST}")
    cache_data = torch.load(CACHE_LATEST, map_location="cpu", weights_only=False)
    for key, _ in splits:
        cache._entries[key] = {
            "z": cache_data[f"{key}_z"], "rho": cache_data[f"{key}_rho"],
            "n": cache_data[f"{key}_n"], "p": noise_rate, "X": cache_data[f"{key}_X"],
        }
else:
    print("BUILDING FORWARD CACHE (once)")
    t0 = time.time()
    for key, A in splits:
        entry = run_batched_safely(cache.compute, key, A, theta_device, forward_circuit,
                                    device=DEVICE, batch_size=128, min_batch=8, p=noise_rate,
                                    label=f"cache[{key}]")
        mb = (entry["rho"].numel()*entry["rho"].element_size() +
              entry["z"].numel()*entry["z"].element_size()) / 1e6
        print(f" [{key:8s}] n={entry['n']:>7,} rho={tuple(entry['rho'].shape)} "
              f"z={tuple(entry['z'].shape)}  {mb:.1f} MB")
    print(f"total: {time.time()-t0:.1f}s")

    save_dict = {"noise_rate": noise_rate, "num_layers": num_layers, "num_qubits": num_qubits}
    for key, _ in splits:
        entry = cache.get(key)
        save_dict[f"{key}_z"]   = entry["z"]
        save_dict[f"{key}_rho"] = entry["rho"]
        save_dict[f"{key}_n"]   = entry["n"]
        save_dict[f"{key}_X"]   = entry["X"]
    torch.save(save_dict, CACHE_LATEST)
    print(f"saved cache -> {CACHE_LATEST} ({CACHE_LATEST.stat().st_size/1e6:.1f} MB)")

for row in cache.memory_report():
    print(f"  {row['key']:8s}: n={row['n_samples']:5d}  {row['MB']:7.1f} MB (CPU)")

QUANTUM CIRCUIT
 device=cpu  wires=4  theta on cpu
BUILDING FORWARD CACHE (once)
 [train   ] n= 81,215 rho=(81215, 16, 16) z=(81215, 4)  334.0 MB
 [test    ] n= 10,086 rho=(10086, 16, 16) z=(10086, 4)  41.5 MB
 [cal     ] n= 10,112 rho=(10112, 16, 16) z=(10112, 4)  41.6 MB
 [zeroday ] n=  1,216 rho=(1216, 16, 16) z=(1216, 4)  5.0 MB
total: 65.7s
saved cache -> /kaggle/working/quantum-sentinel/caches/forward_cache_latest.pt (424.4 MB)
  train   : n=81215    334.0 MB (CPU)
  test    : n=10086     41.5 MB (CPU)
  cal     : n=10112     41.6 MB (CPU)
  zeroday : n= 1216      5.0 MB (CPU)


In [7]:
# ============================================================
# PART 4 — Test-set classification report (cached z -> head)
# ============================================================
y_true_test, y_pred_test = cached_predict_labels(cache.get("test"), y_test, classifier_head,
                                                   device=DEVICE, batch_size=256)
print(classification_report(y_true_test, y_pred_test, target_names=class_names, zero_division=0))

cm = confusion_matrix(y_true_test, y_pred_test, labels=range(num_classes))
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion matrix (test)"); plt.xticks(rotation=60); plt.tight_layout()
savefig("part4_confusion_matrix")

savejson({"classification_report": classification_report(y_true_test, y_pred_test,
          target_names=class_names, zero_division=0, output_dict=True)}, "part4_test_eval")

                precision    recall  f1-score   support

      Analysis       0.06      0.75      0.12        36
      Backdoor       0.01      0.17      0.02        29
           DoS       0.09      0.15      0.11       287
      Exploits       0.55      0.28      0.37      2027
       Fuzzers       0.30      0.36      0.33      1260
       Generic       0.04      0.52      0.07        91
        Normal       0.92      0.51      0.66      5814
Reconnaissance       0.26      0.80      0.39       542

      accuracy                           0.45     10086
     macro avg       0.28      0.44      0.26     10086
  weighted avg       0.70      0.45      0.52     10086

  saved figure -> /kaggle/working/quantum-sentinel/figures/part4_confusion_matrix.png
  saved json -> /kaggle/working/quantum-sentinel/logs/part4_test_eval.json


PosixPath('/kaggle/working/quantum-sentinel/logs/part4_test_eval.json')

In [8]:
# ============================================================
# PART 5 — Global + class-conditional conformal calibration (from cache)
# ============================================================
print(f"min calibration size for alpha={DEFAULT_ALPHA}: n >= {min_calibration_size(DEFAULT_ALPHA):.1f} "
      f"(have n={len(A_cal)})")

q_final, cal_scores_sorted = cached_calibrate_threshold(
    cache.get("cal"), prototypes, alpha=DEFAULT_ALPHA, device=DEVICE, batch_size=256)
print(f"Calibrated global threshold q = {q_final:.4f}")

requested_alphas = (0.01, 0.05, 0.1, 0.2)
feasible_alphas = [a for a in requested_alphas if len(A_cal) >= min_calibration_size(a)]
skipped_alphas = [a for a in requested_alphas if a not in feasible_alphas]
if skipped_alphas:
    print(f"Skipping alpha(s) {skipped_alphas}: n={len(A_cal)} too small (Proposition 3 abstention).")

alpha_rows = cached_conformal_alpha_sweep(cache.get("cal"), cache.get("test"), prototypes,
                                           alphas=feasible_alphas, device=DEVICE, batch_size=256) \
             if feasible_alphas else []
df_alpha = pd.DataFrame(alpha_rows)
auroc_exch = two_sample_discriminability_auroc(A_cal, A_test, seed=SEED)
print(f"cal-vs-test exchangeability AUROC: {auroc_exch:.3f} (want ~0.50)")

if not df_alpha.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(df_alpha["alpha"], df_alpha["empirical_false_alarm_rate"], marker="o", label="empirical FAR")
    ax.plot(df_alpha["alpha"], df_alpha["alpha"], "--", color="black", label="target = alpha")
    ax.set_xlabel("alpha"); ax.set_ylabel("false-alarm rate")
    ax.set_title(f"Proposition 3 alpha-sweep (AUROC={auroc_exch:.3f})"); ax.legend()
    plt.tight_layout(); savefig("part5_alpha_sweep")
savecsv(df_alpha, "part5_alpha_sweep")

# --- Mondrian (per-class) calibration ---
q_by_class, calib_meta = class_conditional_calibrate(
    theta_device, A_cal, y_cal, prototypes, forward_circuit,
    alpha=DEFAULT_ALPHA, device=DEVICE, batch_size=8, fallback="global",
)
for c in sorted(prototypes.keys()):
    m = calib_meta[c]
    print(f" class {c} ({class_names[c]:20s}): n={m['n']:5d} q={m['q']:.4f} status={m['status']}")
print(f" global (marginal) q = {calib_meta['_global']['q']:.4f}")

q_global_map = {c: q_final for c in prototypes.keys()}
far_global   = per_class_empirical_far(theta_device, A_test, y_test, prototypes, forward_circuit,
                                        q_global_map, device=DEVICE, batch_size=8)
far_perclass = per_class_empirical_far(theta_device, A_test, y_test, prototypes, forward_circuit,
                                        q_by_class, device=DEVICE, batch_size=8)
df_far = pd.DataFrame([
    {"class": class_names[r["class"]], "n": r["n"],
     "FAR_global": r["empirical_far"], "FAR_per_class": r2["empirical_far"]}
    for r, r2 in zip(far_global, far_perclass)
])
print(df_far)
savecsv(df_far, "part5_per_class_far")

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_far)); w = 0.35
ax.bar(x - w/2, df_far["FAR_global"], w, label="global", color="steelblue")
ax.bar(x + w/2, df_far["FAR_per_class"], w, label="per-class", color="darkorange")
ax.axhline(DEFAULT_ALPHA, color="black", linestyle="--", label=f"target alpha={DEFAULT_ALPHA}")
ax.set_xticks(x); ax.set_xticklabels(df_far["class"], rotation=60, ha="right")
ax.legend(); plt.tight_layout(); savefig("part5_mondrian_far")

safe_empty_cache()

min calibration size for alpha=0.05: n >= 19.0 (have n=10112)
Calibrated global threshold q = 0.3878
cal-vs-test exchangeability AUROC: 0.494 (want ~0.50)
  saved figure -> /kaggle/working/quantum-sentinel/figures/part5_alpha_sweep.png
  saved table -> /kaggle/working/quantum-sentinel/tables/part5_alpha_sweep.csv
 class 0 (Analysis            ): n=   33 q=0.0484 status=ok
 class 1 (Backdoor            ): n=   31 q=0.4154 status=ok
 class 2 (DoS                 ): n=  284 q=0.4961 status=ok
 class 3 (Exploits            ): n= 2029 q=0.3844 status=ok
 class 4 (Fuzzers             ): n= 1279 q=0.4321 status=ok
 class 5 (Generic             ): n=   92 q=0.4480 status=ok
 class 6 (Normal              ): n= 5820 q=0.3215 status=ok
 class 7 (Reconnaissance      ): n=  544 q=0.0393 status=ok
 global (marginal) q = 0.3878
            class     n  FAR_global  FAR_per_class
0        Analysis    36    0.027778       0.055556
1        Backdoor    29    0.034483       0.034483
2             DoS   28

In [9]:
# ============================================================
# DAY 15 — Depolarizing-channel contraction check on REAL cached rho(x)
# ============================================================
prop1_records = check_proposition1_real_data(
    cache.get("test")["rho"], p_values=(0.0, 0.01, 0.1, 0.3, 0.5, 0.7, 1.0),
    n_pairs=25, seed=SEED,
)
max_resid = assert_proposition1(prop1_records, tol=PROP1_RESIDUAL_TOL)
print(f"Proposition 1 PASSED on real data: max residual = {max_resid:.3e}")

df_prop1 = pd.DataFrame(prop1_records)
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].scatter(df_prop1["D_tr_expected"], df_prop1["D_tr_noisy"], s=10, alpha=0.4)
lims = [0, df_prop1["D_tr_expected"].max() * 1.05]
ax[0].plot(lims, lims, "r--", label="y = x"); ax[0].legend()
ax[0].set_title("Proposition 1: measured vs theoretical")
err_by_p = df_prop1.groupby("p")["residual"].max()
ax[1].plot(err_by_p.index, err_by_p.values, marker="o", color="darkred")
ax[1].set_yscale("log"); ax[1].set_title("Residual vs p (log)")
plt.tight_layout(); savefig("day15_proposition1")
savecsv(df_prop1, "day15_proposition1")

Proposition 1 PASSED on real data: max residual = 6.661e-16
  saved figure -> /kaggle/working/quantum-sentinel/figures/day15_proposition1.png
  saved table -> /kaggle/working/quantum-sentinel/tables/day15_proposition1.csv


PosixPath('/kaggle/working/quantum-sentinel/tables/day15_proposition1.csv')

In [10]:
# ============================================================
# DAY 16 — Analytic L_phi (Lemma 1) + sampled-ratio tightness (from cache)
# ============================================================
L_PHI_ANALYTIC = analytic_lipschitz_bound(num_layers, reupload=reupload)
print(f"Analytic Lipschitz bound: L_phi <= R/2 = {num_layers}/2 = {L_PHI_ANALYTIC:.4f}")

lipschitz_diag = {}
for name in ["train", "test", "cal", "zeroday"]:
    diag = cached_lipschitz_percentile(cache.get(name), n_pairs=min(300, cache.get(name)["n"]//2),
                                        seed=SEED, percentile=95)
    lipschitz_diag[name] = {**diag, **check_lipschitz_tightness(diag, L_PHI_ANALYTIC)}
    status = "OK" if lipschitz_diag[name]["within_bound"] else "!! BOUND VIOLATED !!"
    print(f"{name:8s}: max ratio={lipschitz_diag[name]['max_sampled_ratio']:.4f}  "
          f"gap={lipschitz_diag[name]['tightness_gap']:.4f}  [{status}]")

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (name, res) in zip(axes.flat, lipschitz_diag.items()):
    if len(res["ratios"]) == 0:
        ax.set_title(f"{name}: no data"); continue
    ax.hist(res["ratios"], bins=30, alpha=0.8)
    ax.axvline(L_PHI_ANALYTIC, color="red", linestyle="--", label=f"bound={L_PHI_ANALYTIC:.3f}")
    ax.axvline(res["max"], color="black", linestyle=":", label=f"max={res['max']:.3f}")
    ax.set_title(f"L_phi tightness — {name}"); ax.legend()
plt.tight_layout(); savefig("day16_lipschitz_tightness")

savejson({n: {k: v for k, v in r.items() if k != "ratios"} for n, r in lipschitz_diag.items()},
         "day16_lipschitz")

Analytic Lipschitz bound: L_phi <= R/2 = 2/2 = 1.0000
train   : max ratio=0.7985  gap=0.2015  [OK]
test    : max ratio=0.7978  gap=0.2022  [OK]
cal     : max ratio=0.8086  gap=0.1914  [OK]
zeroday : max ratio=0.7698  gap=0.2302  [OK]
  saved figure -> /kaggle/working/quantum-sentinel/figures/day16_lipschitz_tightness.png
  saved json -> /kaggle/working/quantum-sentinel/logs/day16_lipschitz.json


PosixPath('/kaggle/working/quantum-sentinel/logs/day16_lipschitz.json')

In [11]:
# ============================================================
# Mondrian Proposition-2: per-class F_in_c / F_out_c / Delta_c / eps*_c
# (needs L_PHI_ANALYTIC from Day 16 — run in this order, not before)
# ============================================================
class_ids = sorted(prototypes.keys())
rho_test_all = cache.get("test")["rho"]
rho_zday_all = cache.get("zeroday")["rho"]

per_class_prop2_rows = []
for c in class_ids:
    proto_c = prototypes[c]
    mask_c = (y_test == c); n_c = int(mask_c.sum())
    with torch.no_grad():
        rho_c_test = rho_test_all[mask_c].to(DEVICE)
        proto_b = proto_c.unsqueeze(0).expand(rho_c_test.shape[0], -1, -1)
        f_own_c = fidelity_pairwise(rho_c_test, proto_b).float().cpu().numpy()
        F_in_c = float(f_own_c.min()) if n_c > 0 else float("nan")

        rho_z = rho_zday_all.to(DEVICE)
        proto_b_z = proto_c.unsqueeze(0).expand(rho_z.shape[0], -1, -1)
        f_zday_c = fidelity_pairwise(rho_z, proto_b_z).float().cpu().numpy()
        F_out_c = float(f_zday_c.max())

    delta_c, eps_star_c = proposition2_epsilon_star(F_in_c, F_out_c, p=noise_rate, L_phi=L_PHI_ANALYTIC)
    per_class_prop2_rows.append({"class": class_names[c], "n_known": n_c,
                                  "F_in_c": F_in_c, "F_out_c": F_out_c,
                                  "Delta_c": delta_c, "epsilon_star_c": eps_star_c})
    safe_empty_cache()

df_prop2_perclass = pd.DataFrame(per_class_prop2_rows)
df_prop2_perclass["separable"] = df_prop2_perclass["Delta_c"] > 0
print(df_prop2_perclass.to_string(index=False))
savecsv(df_prop2_perclass, "day18c_mondrian_proposition2")

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["seagreen" if d > 0 else "firebrick" for d in df_prop2_perclass["Delta_c"]]
ax.bar(df_prop2_perclass["class"], df_prop2_perclass["Delta_c"], color=colors)
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); savefig("day18c_mondrian_delta")

         class  n_known   F_in_c  F_out_c   Delta_c  epsilon_star_c  separable
      Analysis       36 0.484956 0.981671 -0.856210       -0.432429      False
      Backdoor       29 0.327071 0.967761 -0.912761       -0.460990      False
           DoS      287 0.362319 0.918802 -0.850856       -0.429725      False
      Exploits     2027 0.289149 0.927343 -0.884627       -0.446781      False
       Fuzzers     1260 0.338518 0.920289 -0.861249       -0.434974      False
       Generic       91 0.495946 0.884385 -0.752739       -0.380171      False
        Normal     5814 0.352397 0.915401 -0.851252       -0.429925      False
Reconnaissance      542 0.457346 0.994253 -0.883542       -0.446233      False
  saved table -> /kaggle/working/quantum-sentinel/tables/day18c_mondrian_proposition2.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day18c_mondrian_delta.png


In [12]:
# ============================================================
# DAY 17 — Certified radius R = m / (2(1-p) L_phi C_f), from cache
# ============================================================
labels_test_final, radii_test_final, scores_test_final, fmaps_test_final = cached_qsnet_infer(
    cache.get("test"), prototypes, q_final, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256,
)
correct_mask = (labels_test_final == y_test)
certified_radii_correct = radii_test_final[correct_mask]
print(f"Correctly-classified & accepted: {correct_mask.sum()} / {len(y_test)}")
print(f"Certified radius: mean={certified_radii_correct.mean():.4f} "
      f"median={np.median(certified_radii_correct):.4f} "
      f"min={certified_radii_correct.min():.4f} max={certified_radii_correct.max():.4f}")

plt.figure(figsize=(7, 4.5))
plt.hist(certified_radii_correct, bins=40, color="teal", alpha=0.8)
plt.axvline(certified_radii_correct.mean(), color="red", linestyle="--",
            label=f"mean={certified_radii_correct.mean():.3f}")
plt.legend(); plt.title("Certified-radius distribution (test set)")
plt.tight_layout(); savefig("day17_certified_radius")

Correctly-classified & accepted: 3254 / 10086
Certified radius: mean=0.0254 median=0.0236 min=0.0000 max=0.0563
  saved figure -> /kaggle/working/quantum-sentinel/figures/day17_certified_radius.png


In [13]:
# ============================================================
# DAY 18 — Proposition 2: F_in, F_out, Delta, epsilon*
# ============================================================
F_max_test    = cached_f_max(cache.get("test"), prototypes, device=DEVICE, batch_size=256)
F_max_zeroday = cached_f_max(cache.get("zeroday"), prototypes, device=DEVICE, batch_size=256)

F_IN  = worst_case_f_in(F_max_test, percentile=0.0)
F_OUT = float(F_max_zeroday.max())
delta_strict, eps_star_strict = proposition2_epsilon_star(F_IN, F_OUT, p=noise_rate, L_phi=L_PHI_ANALYTIC)
F_out_beta, delta_beta, eps_beta = proposition2_epsilon_beta(
    F_IN, F_max_zeroday, L_phi=L_PHI_ANALYTIC, p=noise_rate, beta=DEFAULT_BETA)

print(f"F_in={F_IN:.4f}  F_out={F_OUT:.4f}")
print(f"Delta(strict)={delta_strict:+.4f}  eps*(strict)={eps_star_strict:.4f}"
      f"{' [VACUOUS]' if eps_star_strict <= 0 else ''}")
print(f"F_out^beta={F_out_beta:.4f}  Delta^beta={delta_beta:+.4f}  eps^beta={eps_beta:.4f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].hist(F_max_test, bins=30, alpha=0.6, label="F_max, known", color="seagreen")
ax[0].hist(F_max_zeroday, bins=30, alpha=0.6, label="F_max, zero-day", color="firebrick")
ax[0].axvline(F_IN, color="green", linestyle="--"); ax[0].axvline(F_OUT, color="red", linestyle="--")
ax[0].legend(fontsize=8); ax[0].set_title("F_in vs F_out")
bar_vals = [max(eps_star_strict, 0), max(eps_beta, 0)]
ax[1].bar(["strict", f"beta={DEFAULT_BETA}"], bar_vals,
          color=["slateblue" if eps_star_strict>0 else "lightgray",
                 "darkorange" if eps_beta>0 else "lightgray"])
ax[1].set_title("Certified separable budget epsilon*")
plt.tight_layout(); savefig("day18_proposition2")

F_in=0.3501  F_out=0.9943
Delta(strict)=-0.9310  eps*(strict)=-0.4702 [VACUOUS]
F_out^beta=0.9929  Delta^beta=-0.9296  eps^beta=-0.4695
  saved figure -> /kaggle/working/quantum-sentinel/figures/day18_proposition2.png


In [14]:
# ============================================================
# DAY 18b — Doubly-robust F_in / F_out (percentile-based)
# ============================================================
F_in_robust, F_out_robust, delta_robust, eps_robust = proposition2_epsilon_robust(
    F_max_test, F_max_zeroday, p=noise_rate, L_phi=L_PHI_ANALYTIC,
    lower_percentile=DEFAULT_LOWER_PERCENTILE, upper_percentile=DEFAULT_UPPER_PERCENTILE,
)
df_eps_compare = pd.DataFrame([
    {"variant": "strict", "F_in": F_IN, "F_out": F_OUT, "Delta": delta_strict, "epsilon": eps_star_strict},
    {"variant": f"F_out-relaxed(beta={DEFAULT_BETA})", "F_in": F_IN, "F_out": F_out_beta,
     "Delta": delta_beta, "epsilon": eps_beta},
    {"variant": "doubly-robust(5/97pct)", "F_in": F_in_robust, "F_out": F_out_robust,
     "Delta": delta_robust, "epsilon": eps_robust},
])
print(df_eps_compare.to_string(index=False))
savecsv(df_eps_compare, "day18b_epsilon_variants")

fig, ax = plt.subplots(figsize=(8, 5))
colors = ["slateblue" if e > 0 else "lightgray" for e in df_eps_compare["epsilon"]]
ax.bar(df_eps_compare["variant"], df_eps_compare["epsilon"].clip(lower=0), color=colors)
ax.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=15, ha="right"); plt.tight_layout(); savefig("day18b_epsilon_variants")

                 variant     F_in    F_out     Delta   epsilon
                  strict 0.350135 0.994253 -0.930953 -0.470178
F_out-relaxed(beta=0.05) 0.350135 0.992851 -0.929551 -0.469470
  doubly-robust(5/97pct) 0.612065 0.993358 -0.784165 -0.396043
  saved table -> /kaggle/working/quantum-sentinel/tables/day18b_epsilon_variants.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day18b_epsilon_variants.png


In [15]:
# ============================================================
# HYBRID REJECTION — quantum s(x) fused with classical Isolation Forest
# ============================================================
USE_ISO_HYBRID = True
X_train_classical = X_train  # same feature view fed to the encoder (pre angle-encoding)
X_test_classical, X_cal_classical, X_zeroday_classical = X_test, X_cal, X_zeroday

if USE_ISO_HYBRID:
    iso_clf = IsolationForest(n_estimators=150, contamination="auto", random_state=SEED)
    iso_clf.fit(X_train_classical)
    def iso_nonconformity(X): return -iso_clf.score_samples(X)

    iso_cal, iso_test, iso_zday = (iso_nonconformity(X_cal_classical),
                                    iso_nonconformity(X_test_classical),
                                    iso_nonconformity(X_zeroday_classical))

    q_score_cal  = cached_nonconformity_scores(cache.get("cal"), prototypes, device=DEVICE, batch_size=256)
    q_score_test = cached_nonconformity_scores(cache.get("test"), prototypes, device=DEVICE, batch_size=256)
    q_score_zday = cached_nonconformity_scores(cache.get("zeroday"), prototypes, device=DEVICE, batch_size=256)

    def percentile_rank(cal_ref, values):
        cal_sorted = np.sort(cal_ref)
        return np.searchsorted(cal_sorted, values, side="right") / len(cal_sorted)

    combined_cal  = np.maximum(percentile_rank(q_score_cal, q_score_cal),  percentile_rank(iso_cal, iso_cal))
    combined_test = np.maximum(percentile_rank(q_score_cal, q_score_test), percentile_rank(iso_cal, iso_test))
    combined_zday = np.maximum(percentile_rank(q_score_cal, q_score_zday), percentile_rank(iso_cal, iso_zday))

    q_hybrid, _ = threshold_from_scores(combined_cal, alpha=DEFAULT_ALPHA)
    tp_h, n_h = int(np.sum(combined_zday > q_hybrid)), len(combined_zday)
    fp_h, m_h = int(np.sum(combined_test > q_hybrid)), len(combined_test)
    TPR_hybrid, FPR_hybrid = tp_h/n_h, fp_h/m_h
    tpr_h_lo, tpr_h_hi = clopper_pearson_ci(tp_h, n_h, alpha=0.05)
    fpr_h_lo, fpr_h_hi = clopper_pearson_ci(fp_h, m_h, alpha=0.05)
    print(f"[hybrid] TPR={TPR_hybrid:.4f} CI[{tpr_h_lo:.4f},{tpr_h_hi:.4f}]  "
          f"FPR={FPR_hybrid:.4f} CI[{fpr_h_lo:.4f},{fpr_h_hi:.4f}]")

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].hist(q_score_test, bins=30, alpha=0.6, label="quantum, test")
    ax[0].hist(q_score_zday, bins=30, alpha=0.6, label="quantum, zero-day"); ax[0].legend(fontsize=8)
    ax[1].hist(iso_test, bins=30, alpha=0.6, label="IsoForest, test")
    ax[1].hist(iso_zday, bins=30, alpha=0.6, label="IsoForest, zero-day"); ax[1].legend(fontsize=8)
    plt.tight_layout(); savefig("hybrid_scores")

[hybrid] TPR=0.1291 CI[0.1108,0.1493]  FPR=0.0486 CI[0.0445,0.0530]
  saved figure -> /kaggle/working/quantum-sentinel/figures/hybrid_scores.png


In [16]:
# ============================================================
# DAY 19 — FGSM epsilon sweep (gradient computed ONCE, reused for whole sweep)
# ============================================================
accepted_idx = np.where(labels_test_final != ZERO_DAY)[0][:300]
A_known_accepted = A_test[accepted_idx]
_, proto_stack_main = stack_prototypes(prototypes)

grad_sign_test = run_batched_safely(fgsm_gradient_sign, A_known_accepted, theta_device, prototypes,
                                     forward_circuit, device=DEVICE, proto_stack=proto_stack_main,
                                     batch_size=8, min_batch=1, label="fgsm_grad[test]")

eps_inf_list = [0.0, 0.01, 0.05, 0.08, 0.1, 0.2, 0.3]
day19_rows = []
for eps_inf in eps_inf_list:
    eps_l2 = linf_to_l2_budget(eps_inf, d=num_qubits)
    if eps_inf == 0.0:
        infer_out = cached_qsnet_infer(cache.slice("test", accepted_idx), prototypes, q_final,
                                        p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
                                        zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
    elif grad_sign_test is None:
        infer_out = None
    else:
        X_adv = apply_fgsm_perturbation(A_known_accepted, grad_sign_test, eps_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        infer_out = run_batched_safely(qsnet_infer_batch, X_adv.cpu().numpy(), theta_device, prototypes,
                                        q_final, forward_circuit, p=noise_rate, L_phi=L_PHI_ANALYTIC,
                                        Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                        batch_size=8, min_batch=1, label=f"infer[eps={eps_inf}]")
    if infer_out is None:
        day19_rows.append({"eps_inf": eps_inf, "eps_l2": eps_l2, "acceptance_rate": np.nan}); continue
    day19_rows.append({"eps_inf": eps_inf, "eps_l2": eps_l2,
                        "acceptance_rate": float(np.mean(infer_out[0] != ZERO_DAY))})
    safe_empty_cache()

df_day19 = pd.DataFrame(day19_rows).dropna(subset=["acceptance_rate"])
below = df_day19[df_day19["acceptance_rate"] < 0.99]
empirical_breakpoint_l2 = float(below["eps_l2"].iloc[0]) if len(below) else float("inf")
print(f"Empirical breakpoint (L2, acceptance<99%): {empirical_breakpoint_l2:.4f}")
savecsv(df_day19, "day19_fgsm_sweep")

eps_plot, is_safe = is_eps_safe_to_plot(eps_star_strict)
plt.figure(figsize=(8, 5))
plt.plot(df_day19["eps_l2"], df_day19["acceptance_rate"], marker="o", label="empirical acceptance")
if is_safe:
    plt.axvline(eps_plot, color="green", linestyle="--", label=f"certified eps*={eps_plot:.3f}")
else:
    plt.text(0.02, 0.02, f"eps*={eps_star_strict:.3f} (<=0, not separable)",
              transform=plt.gca().transAxes, color="green")
if np.isfinite(empirical_breakpoint_l2):
    plt.axvline(empirical_breakpoint_l2, color="red", linestyle=":", label="empirical breakpoint")
plt.legend(); plt.title("Day 19: acceptance vs attack budget")
plt.tight_layout(); savefig("day19_fgsm_sweep")

Empirical breakpoint (L2, acceptance<99%): 0.1000
  saved table -> /kaggle/working/quantum-sentinel/tables/day19_fgsm_sweep.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day19_fgsm_sweep.png


In [17]:
# ============================================================
# DAY 20 — H3 (empirical breakpoint >= certified eps*) + noise-rate p sweep
# ============================================================
h3_holds = np.isfinite(empirical_breakpoint_l2) and empirical_breakpoint_l2 >= eps_star_strict
print(f"H3 holds: {h3_holds}" + ("  [VACUOUS: eps*<=0]" if eps_star_strict <= 0 else ""))

p_sweep_values = [0.0,0.05, 0.1, 0.2]
eps_inf_small = [0.0, 0.05, 0.1, 0.2]
day20_rows = []
for p_val in p_sweep_values:
    print(f"p={p_val:.2f}")
    fc_p = theta_dev = prototypes_p = p_cache = None
    try:
        fc_p = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=p_val, reupload=reupload)
        theta_dev = theta_device
        proto_bank_p = PrototypeBank(classes=range(num_classes))
        prototypes_p = run_batched_safely(proto_bank_p.compute, theta_dev, A_train, y_train,
                                           forward_circuit=fc_p, device=DEVICE, batch_size=8,
                                           min_batch=1, label=f"protos[p={p_val}]")
        if prototypes_p is None:
            raise RuntimeError("OOM (prototype computation skipped)")

        p_cache = ForwardCache(store_device="cpu")
        test_entry = run_batched_safely(p_cache.compute, "test", A_test, theta_dev, fc_p, device=DEVICE,
                                         p=p_val, batch_size=8, min_batch=1, label=f"cache_test[p={p_val}]")
        zday_entry = run_batched_safely(p_cache.compute, "zeroday", A_zeroday, theta_dev, fc_p, device=DEVICE,
                                         p=p_val, batch_size=8, min_batch=1, label=f"cache_zday[p={p_val}]")
        if test_entry is None or zday_entry is None:
            raise RuntimeError("OOM (per-p cache build skipped)")

        F_max_test_p = cached_f_max(test_entry, prototypes_p, device=DEVICE, batch_size=256)
        F_max_zday_p = cached_f_max(zday_entry, prototypes_p, device=DEVICE, batch_size=256)
        F_in_p, F_out_p = worst_case_f_in(F_max_test_p, percentile=0.0), float(F_max_zday_p.max())
        _, eps_star_p       = proposition2_epsilon_star(F_in_p, F_out_p, p=p_val, L_phi=L_PHI_ANALYTIC)
        _, eps_star_naive_p = proposition2_epsilon_star(F_IN, F_OUT, p=p_val, L_phi=L_PHI_ANALYTIC)

        labels_base_p, *_ = cached_qsnet_infer(test_entry, prototypes_p, q_final, p=p_val,
                                                L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF, zero_day=ZERO_DAY,
                                                device=DEVICE, batch_size=256)
        accepted_p_idx = np.where(labels_base_p != ZERO_DAY)[0][:200]
        accepted_p = A_test[accepted_p_idx]
        _, proto_stack_p = stack_prototypes(prototypes_p)
        grad_sign_p = run_batched_safely(fgsm_gradient_sign, accepted_p, theta_dev, prototypes_p, fc_p,
                                          device=DEVICE, proto_stack=proto_stack_p, batch_size=8,
                                          min_batch=1, label=f"fgsm_grad[p={p_val}]") if len(accepted_p) else None

        breakpoint_p = float("inf")
        for eps_inf in eps_inf_small:
            if eps_inf == 0.0:
                infer_adv = cached_qsnet_infer(p_cache.slice("test", accepted_p_idx), prototypes_p, q_final,
                                                p=p_val, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
                                                zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
            elif grad_sign_p is None:
                continue
            else:
                X_adv_p = apply_fgsm_perturbation(accepted_p, grad_sign_p, eps_inf, device=DEVICE,
                                                   x_min=0.0, x_max=float(np.pi))
                infer_adv = run_batched_safely(qsnet_infer_batch, X_adv_p.cpu().numpy(), theta_dev,
                                                prototypes_p, q_final, fc_p, p=p_val, L_phi=L_PHI_ANALYTIC,
                                                Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                                batch_size=8, min_batch=1, label=f"infer_adv[p={p_val},eps={eps_inf}]")
            if infer_adv is not None and float(np.mean(infer_adv[0] != ZERO_DAY)) < 0.99:
                breakpoint_p = linf_to_l2_budget(eps_inf, d=num_qubits); break

        day20_rows.append({"p": p_val, "F_in": F_in_p, "F_out": F_out_p,
                            "epsilon_star_correct": eps_star_p, "epsilon_star_naive_wrong": eps_star_naive_p,
                            "empirical_breakpoint_l2": breakpoint_p})
        print(f"  F_in={F_in_p:.4f} F_out={F_out_p:.4f} eps*_correct={eps_star_p:.4f} "
              f"eps*_naive={eps_star_naive_p:.4f} breakpoint={breakpoint_p:.4f}")
    except RuntimeError as e:
        if is_fatal_cuda_error(e):
            print("  [FATAL] non-recoverable CUDA error — stopping p-sweep."); break
        elif is_oom_error(e):
            print(f"  [OOM] p={p_val} skipped."); day20_rows.append({"p": p_val, "F_in": np.nan,
                  "F_out": np.nan, "epsilon_star_correct": np.nan,
                  "epsilon_star_naive_wrong": np.nan, "empirical_breakpoint_l2": np.nan})
        else:
            raise
    finally:
        del fc_p, theta_dev, prototypes_p, p_cache
        safe_empty_cache()

df_day20 = pd.DataFrame(day20_rows)
savecsv(df_day20, "day20_p_sweep")

df_valid = df_day20.dropna(subset=["epsilon_star_correct"])
if not df_valid.empty:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    ax[0].plot(df_valid["p"], df_valid["epsilon_star_correct"], marker="o", label="eps* (re-measured at p)")
    ax[0].plot(df_valid["p"], df_valid["epsilon_star_naive_wrong"], marker="x", linestyle="--",
               color="red", label="eps* (WRONG: fixed at p=0)")
    ax[0].axhline(0, color="gray", linewidth=0.8); ax[0].legend(fontsize=8)
    ax[0].set_title("H4: epsilon* vs p")
    ax[1].plot(df_valid["p"], df_valid["epsilon_star_correct"], marker="o", label="certified eps*")
    ax[1].plot(df_valid["p"], df_valid["empirical_breakpoint_l2"], marker="s", label="empirical breakpoint")
    ax[1].legend(); ax[1].set_title("H3/H4: certified vs empirical")
    plt.tight_layout(); savefig("day20_p_sweep")
    h3_all = (df_valid["empirical_breakpoint_l2"] >= df_valid["epsilon_star_correct"]).all()
    print(f"H3 holds for every p tested: {h3_all}")

H3 holds: True  [VACUOUS: eps*<=0]
p=0.00
  F_in=0.1822 F_out=0.9818 eps*_correct=-0.4825 eps*_naive=-0.4655 breakpoint=0.1000
p=0.05
  F_in=0.6122 F_out=0.9985 eps*_correct=-0.4154 eps*_naive=-0.4900 breakpoint=inf
p=0.10
  F_in=0.7921 F_out=0.9995 eps*_correct=-0.3388 eps*_naive=-0.5172 breakpoint=inf
p=0.20
  F_in=0.9465 F_out=0.9999 eps*_correct=-0.2017 eps*_naive=-0.5818 breakpoint=inf
  saved table -> /kaggle/working/quantum-sentinel/tables/day20_p_sweep.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day20_p_sweep.png
H3 holds for every p tested: True


In [18]:
# ============================================================
# DAY 21 — Extend Prop-2 quantities to train/test/cal + freeze robustness interface
# ============================================================
f_max_cache_by_name = {"test": F_max_test, "zeroday": F_max_zeroday}
day21_rows = [{"dataset": "test", "role": "known", "F_worst_min": float(F_max_test.min()),
               "F_p1": float(np.percentile(F_max_test, 1)), "F_mean": float(F_max_test.mean())}]
for name in ["train", "cal"]:
    fmax = cached_f_max(cache.get(name), prototypes, device=DEVICE, batch_size=256)
    f_max_cache_by_name[name] = fmax
    day21_rows.append({"dataset": name, "role": "known", "F_worst_min": float(fmax.min()),
                        "F_p1": float(np.percentile(fmax, 1)), "F_mean": float(fmax.mean())})
day21_rows.append({"dataset": "zeroday", "role": "novel", "F_worst_max": float(F_max_zeroday.max()),
                    "F_p95": quantile_f_out(F_max_zeroday, beta=0.05), "F_mean": float(F_max_zeroday.mean())})
df_day21 = pd.DataFrame(day21_rows)
savecsv(df_day21, "day21_prop2_all_datasets")

F_IN_GLOBAL = min(f_max_cache_by_name["train"].min(), f_max_cache_by_name["test"].min(),
                   f_max_cache_by_name["cal"].min())
_, EPS_STAR_GLOBAL = proposition2_epsilon_star(F_IN_GLOBAL, F_OUT, p=noise_rate, L_phi=L_PHI_ANALYTIC)
print(f"Global F_in: {F_IN_GLOBAL:.4f}  Global epsilon*: {EPS_STAR_GLOBAL:.4f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot([f_max_cache_by_name[n] for n in ["train", "test", "cal", "zeroday"]],
           labels=["train", "test", "cal", "zeroday"])
ax.axhline(F_IN_GLOBAL, color="green", linestyle="--", label=f"F_in(global)={F_IN_GLOBAL:.3f}")
ax.axhline(F_OUT, color="red", linestyle="--", label=f"F_out(zeroday)={F_OUT:.3f}")
ax.legend(); plt.tight_layout(); savefig("day21_fmax_boxplot")

robustness_interface = {
    "theta": theta_star, "head_state_dict": head_state_dict,
    "prototypes": {c: rho.detach().cpu() for c, rho in prototypes.items()},
    "q_threshold": q_final, "q_by_class": q_by_class, "L_phi_analytic": L_PHI_ANALYTIC,
    "noise_p": noise_rate, "Cf": DEFAULT_CF, "num_qubits": num_qubits, "num_layers": num_layers,
    "F_in_global": F_IN_GLOBAL, "F_out_zeroday_worst": F_OUT, "epsilon_star_global": EPS_STAR_GLOBAL,
    "class_names": class_names, "use_pca": use_pca,
}
torch.save(robustness_interface, CKPT_DIR / "robustness_interface_day21.pt")
print(f"Froze robustness interface -> {CKPT_DIR / 'robustness_interface_day21.pt'}")
safe_empty_cache()

  saved table -> /kaggle/working/quantum-sentinel/tables/day21_prop2_all_datasets.csv
Global F_in: 0.3499  Global epsilon*: -0.4702
  saved figure -> /kaggle/working/quantum-sentinel/figures/day21_fmax_boxplot.png
Froze robustness interface -> /kaggle/working/quantum-sentinel/checkpoints/robustness_interface_day21.pt


In [19]:
# ============================================================
# Hilbert-geometry diagnostics + PCA-of-fidelity-vectors scatter
# ============================================================
h1_report = hilbert_geometry_diagnostics(theta_device, A_train, y_train, prototypes, forward_circuit,
                                          class_names=class_names, device=DEVICE, max_per_class=80,
                                          batch_size=8)
print_h1_report(h1_report)
savejson({k: v for k, v in h1_report.items() if k != "pairs"}, "part16_h1_report")

N_VIZ = 400
rng_viz = np.random.default_rng(SEED)
test_idx_viz = rng_viz.choice(len(A_test), size=min(N_VIZ, len(A_test)), replace=False)
zday_idx_viz = rng_viz.choice(len(A_zeroday), size=min(N_VIZ, len(A_zeroday)), replace=False)
rho_viz = torch.cat([cache.get("test")["rho"][test_idx_viz], cache.get("zeroday")["rho"][zday_idx_viz]], dim=0)
_, feats_viz = fidelity_to_prototypes_matrix(rho_viz, prototypes)
emb_viz = pca_2d(feats_viz)
labels_viz = list(y_test[test_idx_viz]) + [ZERO_DAY] * len(zday_idx_viz)

plt.figure(figsize=(8, 7))
palette = plt.cm.tab10(np.linspace(0, 1, num_classes))
for c in range(num_classes):
    m = np.array([l == c for l in labels_viz])
    plt.scatter(emb_viz[m, 0], emb_viz[m, 1], s=14, alpha=0.7, color=palette[c], label=class_names[c])
m_zday = np.array([l == ZERO_DAY for l in labels_viz])
plt.scatter(emb_viz[m_zday, 0], emb_viz[m_zday, 1], s=20, alpha=0.85, color="black", marker="x", label="zero-day")
plt.legend(fontsize=8); plt.title("PCA of fidelity-to-prototype vectors (test + zero-day)")
plt.tight_layout(); savefig("part16_hilbert_pca_scatter")

=== H1 Hilbert geometry (fidelity gaps) ===
mean intra-class fidelity : 0.8677
mean inter-class fidelity : 0.9494
fidelity gap (intra-inter): -0.0817  ← want ↑
mean inter trace distance : 0.2434  ← want ↑

per-class intra fidelity:
  Analysis                     n=  80  F=0.9473
  Backdoor                     n=  80  F=0.9129
  DoS                          n=  80  F=0.7938
  Exploits                     n=  80  F=0.8686
  Fuzzers                      n=  80  F=0.8171
  Generic                      n=  80  F=0.7892
  Normal                       n=  80  F=0.8436
  Reconnaissance               n=  80  F=0.9691
  saved json -> /kaggle/working/quantum-sentinel/logs/part16_h1_report.json
  saved figure -> /kaggle/working/quantum-sentinel/figures/part16_hilbert_pca_scatter.png


In [20]:
# ============================================================
# DAY 22 — p(x) -> fidelity -> score -> conformal test -> label + radius
# ============================================================
for name, key, idx in [("known-class test sample", "test", 0), ("zeroday sample", "zeroday", 0)]:
    rho_x = cache.get(key)["rho"][idx].to(DEVICE)
    with torch.no_grad():
        f_map = {class_names[c]: float(fidelity(rho_x, prototypes[c])) for c in sorted(prototypes)}
    f_max = max(f_map.values()); score = 1.0 - f_max; accepted = score <= q_final
    decision = max(f_map, key=f_map.get) if accepted else "ZERO_DAY (rejected)"
    print(f"\n--- {name} ---")
    print(f" fidelity: { {k: round(v,4) for k,v in f_map.items()} }")
    print(f" F_max={f_max:.4f}  s(x)={score:.4f}  q={q_final:.4f}  -> {decision}")


--- known-class test sample ---
 fidelity: {'Analysis': 0.7633, 'Backdoor': 0.7048, 'DoS': 0.8028, 'Exploits': 0.816, 'Fuzzers': 0.783, 'Generic': 0.8136, 'Normal': 0.7526, 'Reconnaissance': 0.7028}
 F_max=0.8160  s(x)=0.1840  q=0.3878  -> Exploits

--- zeroday sample ---
 fidelity: {'Analysis': 0.9778, 'Backdoor': 0.9544, 'DoS': 0.9156, 'Exploits': 0.9254, 'Fuzzers': 0.9198, 'Generic': 0.8738, 'Normal': 0.8905, 'Reconnaissance': 0.9851}
 F_max=0.9851  s(x)=0.0149  q=0.3878  -> Reconnaissance


In [21]:
# ============================================================
# DAY 23-24 — RQ1 detection numbers (global threshold, with Clopper-Pearson CIs)
# ============================================================
labels_zday_final, radii_zday_final, scores_zday_final, _ = cached_qsnet_infer(
    cache.get("zeroday"), prototypes, q_final, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)

n_zday = len(labels_zday_final); tp_zday = int(np.sum(labels_zday_final == ZERO_DAY))
TPR_zeroday = tp_zday / n_zday
tpr_lo, tpr_hi = clopper_pearson_ci(tp_zday, n_zday, alpha=0.05)

n_known = len(labels_test_final); fp_known = int(np.sum(labels_test_final == ZERO_DAY))
FPR_known = fp_known / n_known
fpr_lo, fpr_hi = clopper_pearson_ci(fp_known, n_known, alpha=0.05)

accepted_known = labels_test_final != ZERO_DAY
class_acc_known = float(np.mean(labels_test_final[accepted_known] == y_test[accepted_known]))
print(f"[global] TPR={TPR_zeroday:.4f} CI[{tpr_lo:.4f},{tpr_hi:.4f}]  "
      f"FPR={FPR_known:.4f} CI[{fpr_lo:.4f},{fpr_hi:.4f}]  class_acc={class_acc_known:.4f}")

# ============================================================
# DAY 23-24b — RQ1 detection numbers under Mondrian per-class thresholds
# ============================================================
labels_test_pc, radii_test_pc, scores_test_pc, _ = cached_qsnet_infer_per_class(
    cache.get("test"), prototypes, q_by_class, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)
labels_zday_pc, radii_zday_pc, scores_zday_pc, _ = cached_qsnet_infer_per_class(
    cache.get("zeroday"), prototypes, q_by_class, p=noise_rate, L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF,
    zero_day=ZERO_DAY, device=DEVICE, batch_size=256)

tp_pc, n_pc = int(np.sum(labels_zday_pc == ZERO_DAY)), len(labels_zday_pc)
fp_pc, m_pc = int(np.sum(labels_test_pc == ZERO_DAY)), len(labels_test_pc)
TPR_zeroday_pc, FPR_known_pc = tp_pc/n_pc, fp_pc/m_pc
tpr_pc_lo, tpr_pc_hi = clopper_pearson_ci(tp_pc, n_pc, alpha=0.05)
fpr_pc_lo, fpr_pc_hi = clopper_pearson_ci(fp_pc, m_pc, alpha=0.05)
print(f"[per-class] TPR={TPR_zeroday_pc:.4f} CI[{tpr_pc_lo:.4f},{tpr_pc_hi:.4f}]  "
      f"FPR={FPR_known_pc:.4f} CI[{fpr_pc_lo:.4f},{fpr_pc_hi:.4f}]")

# ============================================================
# SECTION 12b — Global vs Mondrian vs Hybrid comparison
# ============================================================
comparison_rows = [
    {"method": "Global", "TPR": TPR_zeroday, "TPR_lo": tpr_lo, "TPR_hi": tpr_hi,
     "FPR": FPR_known, "FPR_lo": fpr_lo, "FPR_hi": fpr_hi},
    {"method": "Mondrian", "TPR": TPR_zeroday_pc, "TPR_lo": tpr_pc_lo, "TPR_hi": tpr_pc_hi,
     "FPR": FPR_known_pc, "FPR_lo": fpr_pc_lo, "FPR_hi": fpr_pc_hi},
]
if USE_ISO_HYBRID:
    comparison_rows.append({"method": "Hybrid", "TPR": TPR_hybrid, "TPR_lo": tpr_h_lo, "TPR_hi": tpr_h_hi,
                             "FPR": FPR_hybrid, "FPR_lo": fpr_h_lo, "FPR_hi": fpr_h_hi})
df_comparison = pd.DataFrame(comparison_rows)
print(df_comparison.to_string(index=False))
savecsv(df_comparison, "part18_global_mondrian_hybrid_comparison")

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(df_comparison)); w = 0.35
tpr_err = [df_comparison["TPR"]-df_comparison["TPR_lo"], df_comparison["TPR_hi"]-df_comparison["TPR"]]
fpr_err = [df_comparison["FPR"]-df_comparison["FPR_lo"], df_comparison["FPR_hi"]-df_comparison["FPR"]]
ax.bar(x-w/2, df_comparison["TPR"], w, yerr=tpr_err, capsize=5, label="TPR", color="seagreen")
ax.bar(x+w/2, df_comparison["FPR"], w, yerr=fpr_err, capsize=5, label="FPR", color="firebrick")
ax.axhline(DEFAULT_ALPHA, color="black", linestyle="--", label=f"alpha={DEFAULT_ALPHA}")
ax.set_xticks(x); ax.set_xticklabels(df_comparison["method"]); ax.set_ylim(0,1.05); ax.legend()
plt.tight_layout(); savefig("part18_comparison")
safe_empty_cache()

[global] TPR=0.2377 CI[0.2140,0.2626]  FPR=0.0503 CI[0.0461,0.0547]  class_acc=0.3397
[per-class] TPR=0.0896 CI[0.0742,0.1071]  FPR=0.2149 CI[0.2069,0.2230]
  method      TPR   TPR_lo   TPR_hi      FPR   FPR_lo   FPR_hi
  Global 0.237664 0.213987 0.262618 0.050268 0.046084 0.054713
Mondrian 0.089638 0.074179 0.107114 0.214852 0.206870 0.222999
  Hybrid 0.129112 0.110768 0.149270 0.048582 0.044468 0.052960
  saved table -> /kaggle/working/quantum-sentinel/tables/part18_global_mondrian_hybrid_comparison.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/part18_comparison.png


In [22]:
# ============================================================
# DAY 25 — RQ3: disentangling adversarial-known from true zero-day
# ============================================================
eps_l2_sweep = np.linspace(0.0, max(abs(EPS_STAR_GLOBAL)*2.5, 1.0), 8)
eps_inf_sweep = [l2_to_linf_budget(e, d=num_qubits) for e in eps_l2_sweep]
rq3_idx = np.where(accepted_known)[0][:250]
A_known_rq3 = A_test[rq3_idx]
grad_sign_rq3 = run_batched_safely(fgsm_gradient_sign, A_known_rq3, theta_device, prototypes,
                                    forward_circuit, device=DEVICE, proto_stack=proto_stack_main,
                                    batch_size=8, min_batch=1, label="fgsm_grad[rq3]")

rq3_rows = []
for e_l2, e_inf in zip(eps_l2_sweep, eps_inf_sweep):
    if e_inf == 0.0:
        infer_out = cached_qsnet_infer(cache.slice("test", rq3_idx), prototypes, q_final, p=noise_rate,
                                        L_phi=L_PHI_ANALYTIC, Cf=DEFAULT_CF, zero_day=ZERO_DAY,
                                        device=DEVICE, batch_size=256)
    elif grad_sign_rq3 is None:
        infer_out = None
    else:
        X_adv = apply_fgsm_perturbation(A_known_rq3, grad_sign_rq3, e_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        infer_out = run_batched_safely(qsnet_infer_batch, X_adv.cpu().numpy(), theta_device, prototypes,
                                        q_final, forward_circuit, p=noise_rate, L_phi=L_PHI_ANALYTIC,
                                        Cf=DEFAULT_CF, zero_day=ZERO_DAY, device=DEVICE,
                                        batch_size=8, min_batch=1, label=f"rq3_infer[eps={e_inf:.3f}]")
    reject_rate = float(np.mean(infer_out[0] == ZERO_DAY)) if infer_out is not None else np.nan
    rq3_rows.append({"eps_l2": e_l2, "adv_known_reject_rate": reject_rate})
    safe_empty_cache()

df_rq3 = pd.DataFrame(rq3_rows).dropna(subset=["adv_known_reject_rate"])
savecsv(df_rq3, "day25_rq3_disentanglement")

if not df_rq3.empty:
    eps_plot, is_safe = is_eps_safe_to_plot(EPS_STAR_GLOBAL)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_rq3["eps_l2"], df_rq3["adv_known_reject_rate"], marker="o", label="adversarial-known reject rate")
    ax.axhline(TPR_zeroday, color="firebrick", linestyle="--", label=f"true zero-day reject rate ({TPR_zeroday:.2f})")
    if is_safe:
        ax.axvline(eps_plot, color="green", linestyle=":", label=f"predicted eps*={eps_plot:.3f}")
    ax.set_ylim(-0.02, 1.05); ax.legend(); ax.set_title("RQ3: disentangling adversarial-known from zero-day")
    plt.tight_layout(); savefig("day25_rq3_disentanglement")

  saved table -> /kaggle/working/quantum-sentinel/tables/day25_rq3_disentanglement.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day25_rq3_disentanglement.png


In [23]:
# ============================================================
# Save the full Days 15-25 results bundle
# ============================================================
theory_validation = {
    "prop1_max_residual": max_resid, "lipschitz_analytic": L_PHI_ANALYTIC,
    "certified_radius": {"mean": float(certified_radii_correct.mean()),
                          "median": float(np.median(certified_radii_correct)),
                          "min": float(certified_radii_correct.min()),
                          "max": float(certified_radii_correct.max())},
    "F_in": F_IN, "F_out": F_OUT, "epsilon_star_strict": eps_star_strict,
    "epsilon_star_strict_is_vacuous": bool(eps_star_strict <= 0),
    "epsilon_beta": eps_beta, "doubly_robust_epsilon": eps_robust,
    "per_class_prop2": df_prop2_perclass.to_dict(orient="records"),
    "day19_acceptance_curve": df_day19.to_dict(orient="records"),
    "empirical_breakpoint_l2": empirical_breakpoint_l2, "H3_holds": bool(h3_holds),
    "day20_p_sweep": df_day20.to_dict(orient="records"),
    "day21_per_dataset": df_day21.to_dict(orient="records"),
    "F_in_global": float(F_IN_GLOBAL), "epsilon_star_global": EPS_STAR_GLOBAL,
    "RQ1_detection_global": {"TPR": TPR_zeroday, "TPR_ci95": [tpr_lo, tpr_hi],
                              "FPR": FPR_known, "FPR_ci95": [fpr_lo, fpr_hi]},
    "RQ1_detection_mondrian": {"TPR": TPR_zeroday_pc, "TPR_ci95": [tpr_pc_lo, tpr_pc_hi],
                                "FPR": FPR_known_pc, "FPR_ci95": [fpr_pc_lo, fpr_pc_hi]},
    "comparison_table": df_comparison.to_dict(orient="records"),
    "RQ3_disentanglement": df_rq3.to_dict(orient="records"),
    "cache_memory_report": cache.memory_report(),
}
savejson(theory_validation, "theory_validation_days15_25")

  saved json -> /kaggle/working/quantum-sentinel/logs/theory_validation_days15_25.json


PosixPath('/kaggle/working/quantum-sentinel/logs/theory_validation_days15_25.json')

In [24]:
# ============================================================
# DAY 26 (a) — Certified-accuracy-at-radius curve (RQ4)
# ============================================================
radius_grid = np.linspace(0, np.percentile(certified_radii_correct, 99), 40)
cert_acc = [float(np.mean((labels_test_final == y_test) & (radii_test_final >= r))) for r in radius_grid]
df_cert_acc = pd.DataFrame({"radius": radius_grid, "certified_accuracy": cert_acc})
savecsv(df_cert_acc, "day26_certified_accuracy_at_radius")

plt.figure(figsize=(7.5, 5))
plt.plot(radius_grid, cert_acc, marker=".", color="teal")
plt.xlabel("certified radius r"); plt.ylabel("certified accuracy (fraction with R>=r, correctly labeled)")
plt.title("RQ4: certified-accuracy-at-radius"); plt.tight_layout()
savefig("day26_certified_accuracy_curve")

# ============================================================
# DAY 26 (b) — Noise-rate p ablation (certified vs empirical), reusing Day 20
# ============================================================
df_noise_ablation = df_day20.copy()
df_noise_ablation["h3_holds_row"] = df_noise_ablation["empirical_breakpoint_l2"] >= df_noise_ablation["epsilon_star_correct"]
savecsv(df_noise_ablation, "day26_noise_rate_ablation")
print(df_noise_ablation.to_string(index=False))

  saved table -> /kaggle/working/quantum-sentinel/tables/day26_certified_accuracy_at_radius.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/day26_certified_accuracy_curve.png
  saved table -> /kaggle/working/quantum-sentinel/tables/day26_noise_rate_ablation.csv
   p     F_in    F_out  epsilon_star_correct  epsilon_star_naive_wrong  empirical_breakpoint_l2  h3_holds_row
0.00 0.182207 0.981806             -0.482533                 -0.465476                      0.1          True
0.05 0.612199 0.998509             -0.415375                 -0.489975                      inf          True
0.10 0.792095 0.999477             -0.338819                 -0.517196                      inf          True
0.20 0.946455 0.999902             -0.201712                 -0.581845                      inf          True


In [25]:
# ============================================================
# DAY 26 (c) — MAQT ablation: CE-only baseline vs MAQT
# *** THIS BLOCK TRAINS A NEW MODEL (train_plain_vqc). ***
# Everything else in this notebook reuses your existing MAQT checkpoint;
# this is the one exception, because there is no CE-only checkpoint saved yet.
# ============================================================
RUN_MAQT_ABLATION = False   # <-- flip to True only when you want to spend the compute

if RUN_MAQT_ABLATION:
    PLAIN_CKPT_DIR = CKPT_DIR / "plain_vqc"
    theta_plain, head_plain, history_plain = train_plain_vqc(
        A_train, y_train, n_classes=num_classes, n_qubits=num_qubits, n_layers=num_layers,
        forward_circuit=forward_circuit, device=DEVICE,
        epochs=10, lr=0.05, batch_size=8, use_weighted_sampler=True,
        checkpoint_dir=str(PLAIN_CKPT_DIR), log_dir=str(LOG_DIR), notebook_name="plain_vqc",
        seed=SEED, verbose=True,
    )
    torch.save({"theta": theta_plain.detach().cpu(),
                "head_state_dict": head_plain.state_dict(),
                "history": history_plain}, CKPT_DIR / "plain_vqc_final.pt")

    epsilons = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
    ablation_rows = robustness_ablation(
        A_test, y_test, theta_device, classifier_head, theta_plain.to(DEVICE), head_plain,
        forward_circuit, epsilons, DEVICE, attack_fn=fgsm_attack, batch_size=256,
        x_min=0.0, x_max=float(np.pi),
    )
    df_maqt_ablation = pd.DataFrame(ablation_rows)
    savecsv(df_maqt_ablation, "day26_maqt_vs_ce_ablation")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_maqt_ablation["eps"], df_maqt_ablation["maqt_acc"], marker="o", label="MAQT")
    ax.plot(df_maqt_ablation["eps"], df_maqt_ablation["plain_acc"], marker="s", label="CE-only")
    ax.set_xlabel("FGSM epsilon"); ax.set_ylabel("accuracy"); ax.legend()
    ax.set_title("MAQT ablation: geometry-shaping benefit under attack")
    plt.tight_layout(); savefig("day26_maqt_vs_ce_ablation")
else:
    print("MAQT ablation skipped (RUN_MAQT_ABLATION=False) — set True to train the CE-only baseline.")

MAQT ablation skipped (RUN_MAQT_ABLATION=False) — set True to train the CE-only baseline.


In [26]:
# ============================================================
# DAY 26 (d) — Freeze all artifacts + one-command re-run confirmation
# ============================================================
frozen_artifacts = {
    "theta": theta_star, "head_state_dict": head_state_dict,
    "prototypes": {c: rho.detach().cpu() for c, rho in prototypes.items()},
    "q_threshold": q_final, "q_by_class": q_by_class, "L_phi_analytic": L_PHI_ANALYTIC,
    "noise_p": noise_rate, "num_qubits": num_qubits, "num_layers": num_layers,
    "class_names": class_names, "checkpoint_source": str(CHECKPOINT_PATH),
    "cache_source": str(CACHE_LATEST), "frozen_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
FROZEN_PATH = CKPT_DIR / "qsnet_frozen_day26.pt"
torch.save(frozen_artifacts, FROZEN_PATH)
print(f"Frozen artifacts -> {FROZEN_PATH}")

# one-command re-run check: reload from disk and re-verify a single known/zero-day sample
# _frozen = torch.load(FROZEN_PATH, map_location="cpu", weights_only=False)
# _protos_reload = {c: v.to(DEVICE) for c, v in _frozen["prototypes"].items()}
# rho_chk = cache.get("test")["rho"][0].to(DEVICE)
# with torch.no_grad():
#     f_chk = {c: float(fidelity(rho_chk, _protos_reload[c])) for c in _protos_reload}
# print("Re-run check (fidelities from reloaded frozen artifacts):", f_chk)

Frozen artifacts -> /kaggle/working/quantum-sentinel/checkpoints/qsnet_frozen_day26.pt


In [27]:
# ============================================================
# DAY 27 (a) — Figure 1: trust-region visualisation
# (2D projection of Hilbert-space prototypes + samples)
# ============================================================
N_VIZ2 = 600
rng2 = np.random.default_rng(SEED)
idx_t = rng2.choice(len(A_test), size=min(N_VIZ2, len(A_test)), replace=False)
idx_z = rng2.choice(len(A_zeroday), size=min(N_VIZ2, len(A_zeroday)), replace=False)

rho_known = cache.get("test")["rho"][idx_t]
rho_zday  = cache.get("zeroday")["rho"][idx_z]
proto_stack_t = torch.stack([prototypes[c].cpu() for c in sorted(prototypes)], dim=0)
rho_all_fig1 = torch.cat([rho_known, rho_zday, proto_stack_t], dim=0)

_, feats_fig1 = fidelity_to_prototypes_matrix(rho_all_fig1, prototypes)
emb_fig1 = pca_2d(feats_fig1)
n_known, n_zday_, n_proto = len(idx_t), len(idx_z), proto_stack_t.shape[0]
emb_known, emb_zday, emb_proto = emb_fig1[:n_known], emb_fig1[n_known:n_known+n_zday_], emb_fig1[n_known+n_zday_:]

labels_known_fig1 = y_test[idx_t]
plt.figure(figsize=(9, 8))
palette = plt.cm.tab10(np.linspace(0, 1, num_classes))
for c in range(num_classes):
    m = labels_known_fig1 == c
    plt.scatter(emb_known[m, 0], emb_known[m, 1], s=12, alpha=0.5, color=palette[c], label=class_names[c])
plt.scatter(emb_zday[:, 0], emb_zday[:, 1], s=16, alpha=0.6, color="black", marker="x", label="zero-day")
for i, c in enumerate(sorted(prototypes)):
    plt.scatter(*emb_proto[i], s=260, marker="*", color=palette[c], edgecolor="black", linewidth=1.2, zorder=5)
plt.legend(fontsize=8, loc="best")
plt.title("Figure 1 — Trust-region visualisation\n(2D PCA of Hilbert-space fidelity vectors; stars = class prototypes)")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout()
savefig("figure1_trust_regions")

# ============================================================
# DAY 27 (b) — Figure 3: disentanglement (separation AUROC vs attack budget)
# ============================================================
auroc_rows = []
for e_l2, e_inf in zip(eps_l2_sweep, eps_inf_sweep):
    if e_inf == 0.0:
        rho_known_adv = cache.slice("test", rq3_idx)["rho"]
    elif grad_sign_rq3 is None:
        continue
    else:
        X_adv = apply_fgsm_perturbation(A_known_rq3, grad_sign_rq3, e_inf, device=DEVICE,
                                         x_min=0.0, x_max=float(np.pi))
        with torch.no_grad():
            _, rho_known_adv = forward_circuit(X_adv, theta_device)
        rho_known_adv = rho_known_adv.detach().cpu()

    s_known = 1.0 - np.array([
        max(float(fidelity(rho_known_adv[i].to(DEVICE), prototypes[c])) for c in prototypes)
        for i in range(rho_known_adv.shape[0])
    ])
    s_zday = scores_zday_final  # from Day 23-24, fixed zero-day nonconformity scores

    y_bin = np.concatenate([np.zeros(len(s_known)), np.ones(len(s_zday))])
    s_all = np.concatenate([s_known, s_zday])
    sep_auroc = roc_auc_score(y_bin, s_all)
    auroc_rows.append({"eps_l2": e_l2, "separation_auroc": sep_auroc})

df_fig3 = pd.DataFrame(auroc_rows)
savecsv(df_fig3, "figure3_disentanglement_auroc")

plt.figure(figsize=(8, 5))
plt.plot(df_fig3["eps_l2"], df_fig3["separation_auroc"], marker="o", color="darkorange")
plt.axhline(0.5, color="gray", linestyle=":", label="chance (0.5)")
eps_plot, is_safe = is_eps_safe_to_plot(EPS_STAR_GLOBAL)
if is_safe:
    plt.axvline(eps_plot, color="green", linestyle="--", label=f"Proposition-2 threshold eps*={eps_plot:.3f}")
plt.ylim(0.4, 1.02); plt.xlabel("attack budget epsilon (L2)"); plt.ylabel("separation AUROC (known-adv vs zero-day)")
plt.title("Figure 3 — Disentanglement: separation AUROC vs attack budget")
plt.legend(); plt.tight_layout()
savefig("figure3_disentanglement")

  saved figure -> /kaggle/working/quantum-sentinel/figures/figure1_trust_regions.png
  saved table -> /kaggle/working/quantum-sentinel/tables/figure3_disentanglement_auroc.csv
  saved figure -> /kaggle/working/quantum-sentinel/figures/figure3_disentanglement.png


In [28]:
# ============================================================
# DAY 28 (a) — Table C: certified + empirical accuracy under FGSM/PGD, across eps and noise p
# ============================================================
epsilons_tableC = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2]
p_values_tableC = [0.0, noise_rate, 0.1, 0.2]

table_c_rows = []
for p_val in p_values_tableC:
    fc_p = build_forward_circuit(dev, num_qubits, num_layers, noise_rate=p_val, reupload=reupload)
    for eps in epsilons_tableC:
        # empirical accuracy under FGSM (classifier-head accuracy on perturbed known samples)
        res_fgsm = eval_attacked(fgsm_attack, A_test, y_test, theta_device, classifier_head, fc_p,
                                  DEVICE, batch_size=32, eps=eps, x_min=torch.tensor(0.0, device=DEVICE), x_max=torch.tensor(np.pi, device=DEVICE))
        # certified accuracy at this eps (radius >= eps, correct label), from the p=noise_rate certificate
        cert_acc_eps = float(np.mean((labels_test_final == y_test) & (radii_test_final >= eps)))
        table_c_rows.append({"p": p_val, "eps": eps,
                              "empirical_acc_fgsm": res_fgsm["acc"],
                              "empirical_macro_f1_fgsm": res_fgsm["macro_f1"],
                              "certified_acc": cert_acc_eps})
    safe_empty_cache()

df_table_c = pd.DataFrame(table_c_rows)
savecsv(df_table_c, "day28_table_c_robustness")
print(df_table_c.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5.5))
for p_val, grp in df_table_c.groupby("p"):
    ax.plot(grp["eps"], grp["empirical_acc_fgsm"], marker="o", label=f"empirical (p={p_val:.2f})")
ax.plot(df_table_c.query("p == @noise_rate")["eps"],
        df_table_c.query("p == @noise_rate")["certified_acc"],
        marker="s", linestyle="--", color="black", label="certified (p=trained noise)")
ax.set_xlabel("attack budget epsilon"); ax.set_ylabel("accuracy")
ax.set_title("Table C — certified vs empirical accuracy across eps and noise p")
ax.legend(fontsize=8); plt.tight_layout()
savefig("day28_table_c_robustness")

# ============================================================
# DAY 28 (b) — Certified-radius distribution plots (all datasets that have labels)
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(certified_radii_correct, bins=40, alpha=0.8, color="teal", label="test (correct & accepted)")
ax.axvline(certified_radii_correct.mean(), color="red", linestyle="--",
           label=f"mean={certified_radii_correct.mean():.3f}")
ax.axvline(np.median(certified_radii_correct), color="orange", linestyle=":",
           label=f"median={np.median(certified_radii_correct):.3f}")
ax.set_xlabel("certified radius R"); ax.set_title("Day 28 — final certified-radius distribution")
ax.legend(); plt.tight_layout()
savefig("day28_certified_radius_final")

  saved table -> /kaggle/working/quantum-sentinel/tables/day28_table_c_robustness.csv
   p  eps  empirical_acc_fgsm  empirical_macro_f1_fgsm  certified_acc
0.00 0.00            0.452310                 0.257865       0.322625
0.00 0.02            0.397977                 0.199710       0.172120
0.00 0.05            0.349693                 0.153557       0.031925
0.00 0.10            0.232897                 0.077305       0.000000
0.00 0.15            0.212572                 0.065866       0.000000
0.00 0.20            0.169740                 0.053808       0.000000
0.01 0.00            0.451814                 0.257875       0.322625
0.01 0.02            0.402736                 0.205097       0.172120
0.01 0.05            0.347908                 0.153201       0.031925
0.01 0.10            0.253321                 0.084446       0.000000
0.01 0.15            0.215249                 0.066741       0.000000
0.01 0.20            0.173706                 0.054393       0.000000
0.10

In [29]:
# ============================================================
# Save the Day 26-28 results bundle
# ============================================================
final_bundle = {
    "day26_certified_accuracy_curve": df_cert_acc.to_dict(orient="records"),
    "day26_noise_rate_ablation": df_noise_ablation.to_dict(orient="records"),
    "day26_maqt_ablation_ran": bool(RUN_MAQT_ABLATION),
    "day27_figure3_auroc": df_fig3.to_dict(orient="records"),
    "day28_table_c": df_table_c.to_dict(orient="records"),
    "frozen_artifacts_path": str(FROZEN_PATH),
}
savejson(final_bundle, "results_days26_28")
print("\nAll outputs written under:", OUT_ROOT)
print(" figures  ->", FIG_DIR)
print(" tables   ->", TABLE_DIR)
print(" logs     ->", LOG_DIR)
print(" ckpts    ->", CKPT_DIR)

  saved json -> /kaggle/working/quantum-sentinel/logs/results_days26_28.json

All outputs written under: /kaggle/working/quantum-sentinel
 figures  -> /kaggle/working/quantum-sentinel/figures
 tables   -> /kaggle/working/quantum-sentinel/tables
 logs     -> /kaggle/working/quantum-sentinel/logs
 ckpts    -> /kaggle/working/quantum-sentinel/checkpoints
